In [33]:
!pip install pycldf pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
!git clone https://github.com/grambank/grambank.git

fatal: destination path 'grambank' already exists and is not an empty directory.


In [35]:
from pycldf import Dataset
import pandas as pd

In [36]:
def load_language_data():
    dataset = Dataset.from_metadata("grambank/cldf/StructureDataset-metadata.json")

    values = list(dataset["ValueTable"])

    # 3. Convert to a Pandas DataFrame for easy filtering and analysis
    df = pd.DataFrame([
        {
            "Language_ID": row["Language_ID"], # The Glottocode (e.g., 'stan1293' for English)
            "Feature_ID": row["Parameter_ID"], # The Grambank feature (e.g., 'GB024')
            "Value": row["Value"],             # The binary score (0, 1, or ?)
            "Source": row["Source"]            # The bibliographic reference
        }
        for row in values
    ])

    wide_df = df.pivot(index="Language_ID", columns="Feature_ID", values="Value")

    wide_df.reset_index(inplace=True)

    url = "https://raw.githubusercontent.com/glottolog/glottolog-cldf/master/cldf/languages.csv"
    glottolog_df = pd.read_csv(url)

    mapping_df = glottolog_df[['ID', 'ISO639P3code', 'Name']].rename(columns={
        'ID': 'Language_ID', 
        'ISO639P3code': 'ISO639_3',
        'Name': 'Language_Name'
    })

    mapping_df = mapping_df.dropna(subset=['ISO639_3'])

    language_codes_df = pd.merge(mapping_df, wide_df, on="Language_ID", how="inner")

    language_codes_df.drop(columns=["Language_ID"], inplace=True)

    language_codes_df.to_csv("grambank_language_values.csv", index=False)

In [ ]:
def load_parameter_data():
    dataset = Dataset.from_metadata("grambank/cldf/StructureDataset-metadata.json")
    
    parameters = list(dataset["ParameterTable"])

    param_df = pd.DataFrame([
        {
            "Feature_ID": param["ID"],
            "Question": param.get("Name"),
            "Definition": param.get("Description")
        }
        for param in parameters
    ])

    print(param_df.head())

    param_df.rename(columns={"ISO639_3": "ISO_Code"}, inplace=True)

    param_df.to_csv("grambank_parameters.csv", index=False)

In [38]:
def load_grambank_data():
    load_language_data()
    load_parameter_data()

load_grambank_data()
    

  Feature_ID                                           Question  \
0      GB020           Are there definite or specific articles?   
1      GB021  Do indefinite nominals commonly have indefinit...   
2      GB022                     Are there prenominal articles?   
3      GB023                    Are there postnominal articles?   
4      GB024   What is the order of numeral and noun in the NP?   

                                          Definition  
0  ## Are there definite or specific articles?\r\...  
1  ## Do indefinite/non-specific nominals commonl...  
2  ## Are there prenominal articles?\r\n\r\n## Su...  
3  ## Are there postnominal articles?\r\n\r\n## S...  
4  ## What is the order of numeral and noun in th...  


In [ ]:
def get_question(feature_id):
    features_df = pd.read_csv("grambank_parameters.csv")
    for row in features_df.itertuples():
        if row.Feature_ID == feature_id:
            return row.Question
    return None

: 